<a href="https://colab.research.google.com/github/Shreya-Reddy-7/AI-Powered-Intelligent-Recruitment-Candidate-Analytics-System/blob/job_analyzer_module/Job_analyser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import nltk
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/AI_Genesis_Dataset/job_title_des.csv')

df.head()

,Unnamed: 0,Job Title,Job Description
0,0,Flutter Developer,We are looking for hire experts flutter develo...
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."
3,3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...
4,4,Full Stack Developer,job responsibility full stack engineer – react...


In [ ]:
df = df.drop(columns=['Unnamed: 0'])

In [ ]:
df.columns = ['job_title', 'job_description']

In [ ]:
df.head()

,job_title,job_description
0,Flutter Developer,We are looking for hire experts flutter develo...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...
4,Full Stack Developer,job responsibility full stack engineer – react...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2277 entries, 0 to 2276
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   job_title        2277 non-null   object
 1   job_description  2277 non-null   object
dtypes: object(2)
memory usage: 35.7+ KB


In [ ]:
df['job_description'] = df['job_description'].fillna('')

In [ ]:
import nltk
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
def clean_text(text):

    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]','',text)   # remove numbers & symbols

    words = text.split()
    words = [w for w in words if w not in stopwords.words('english')]

    return " ".join(words)

In [ ]:
df['clean_description'] = df['job_description'].apply(clean_text)

df[['job_title','clean_description']].head()

,job_title,clean_description
0,Flutter Developer,looking hire experts flutter developer eligibl...
1,Django Developer,pythondjango developerlead job codepdj strong ...
2,Machine Learning,data scientist contractorbangalore inresponsib...
3,iOS Developer,job descriptionstrong framework outside ios al...
4,Full Stack Developer,job responsibility full stack engineer react r...


In [ ]:
vectorizer = TfidfVectorizer(max_features=1000)

job_vectors = vectorizer.fit_transform(df['clean_description'])

print(job_vectors.shape)

(2277, 1000)


In [ ]:
feature_names = vectorizer.get_feature_names_out()

def get_keywords(vector, top_n=5):

    sorted_items = vector.toarray().argsort()[0][-top_n:]
    return [feature_names[i] for i in sorted_items]


for i in range(5):

    keywords = get_keywords(job_vectors[i])

    print("Job:", df['job_title'][i])
    print("Keywords:", keywords)
    print()

Job: Flutter Developer
Keywords: ['covid', 'remotelytemporarily', 'flutter', 'hire', 'shiftsupplemental']

Job: Django Developer
Keywords: ['improving', 'handle', 'efficiency', 'programs', 'api']

Job: Machine Learning
Keywords: ['machine', 'big', 'data', 'learning', 'scientist']

Job: iOS Developer
Keywords: ['working', 'excellent', 'mobile', 'core', 'ios']

Job: Full Stack Developer
Keywords: ['concept', 'across', 'mobile', 'react', 'web']



In [ ]:
import pandas as pd
import re
import json
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/AI_Genesis_Dataset/job_title_des.csv')

df = df.drop(columns=['Unnamed: 0'])
df.columns = ['job_title','job_description']

df.head()

,job_title,job_description
0,Flutter Developer,We are looking for hire experts flutter develo...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...
4,Full Stack Developer,job responsibility full stack engineer – react...


In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]',' ',text)

    words = text.split()
    words = [w for w in words if w not in stop_words]

    return " ".join(words)

df['clean_description'] = df['job_description'].apply(clean_text)

In [ ]:
skills_list = [

'python','java','c++','sql','machine learning','deep learning','tensorflow',
'pytorch','nlp','data analysis','data science','flask','django','react',
'node','flutter','android','ios','docker','kubernetes','aws','git',
'communication','teamwork','problem solving','leadership','critical thinking'

]

In [ ]:
def extract_skills(text):

    found_skills = []

    for skill in skills_list:
        if skill in text:
            found_skills.append(skill)

    return found_skills

df['skills'] = df['clean_description'].apply(extract_skills)

In [ ]:
def extract_experience(text):

    exp = re.findall(r'\d+\+?\s*years?', text)

    if exp:
        return exp[0]
    else:
        return "Not Specified"

df['experience_required'] = df['job_description'].apply(extract_experience)

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000)

tfidf_matrix = vectorizer.fit_transform(df['clean_description'])

feature_names = vectorizer.get_feature_names_out()

In [ ]:
job_data = []

for i,row in df.iterrows():

    job = {
        "job_title": row['job_title'],
        "skills": row['skills'],
        "experience_required": row['experience_required']
    }

    job_data.append(job)

print(json.dumps(job_data[:5], indent=4))

[
    {
        "job_title": "Flutter Developer",
        "skills": [
            "flutter"
        ],
        "experience_required": "1 year"
    },
    {
        "job_title": "Django Developer",
        "skills": [
            "python",
            "sql",
            "flask",
            "django",
            "communication"
        ],
        "experience_required": "Not Specified"
    },
    {
        "job_title": "Machine Learning",
        "skills": [
            "python",
            "java",
            "machine learning",
            "deep learning",
            "tensorflow",
            "pytorch",
            "communication"
        ],
        "experience_required": "3 years"
    },
    {
        "job_title": "iOS Developer",
        "skills": [
            "ios"
        ],
        "experience_required": "Not Specified"
    },
    {
        "job_title": "Full Stack Developer",
        "skills": [
            "java",
            "react",
            "git",
            "communica

In [ ]:
with open("structured_jobs.json","w") as f:
    json.dump(job_data,f,indent=4)

In [ ]:
!ls

drive  sample_data  structured_jobs.json


In [ ]:
with open('/content/drive/MyDrive/AI_Genesis_Dataset/structured_jobs.json','w') as f:
    json.dump(job_data,f,indent=4)

In [ ]:
from google.colab import files
files.download('structured_jobs.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>